# Shelf List Report

This notebook will create a CSV file of Inventory Items with the following information: Barcode, Title, Effective Location, Effective Call Number Components, Material Type, and Item Status

## 1. Environment setup

In [ ]:
import pandas as pd
import requests
from datetime import datetime, date

pd.set_option('display.max_columns', None)

## 2. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [ ]:
%run folio_auth.ipynb

## 3. Helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. 

In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

## 4. Get Locations

In [ ]:
locs_raw = fetch_all_records(
    "/locations",
    records_key="locations",
    limit=50,
    query='isActive=="true"',
) 
print(f"{len(locs_raw)} active locations found")
locs_df = pd.DataFrame(locs_raw)
print(locs_df.loc[:, ['name','id']].sort_values(by='name'))


## 5. Retrieve Item records from the Selected Effective Location
***Note:*** Not intended for very large locations (200,000 or more). 

In [ ]:
locationId = 'REPLACE WITH THE LOCATION ID FROM ABOVE'

searchLoc = locs_df.loc[locs_df['id'] == locationId]
locationName = searchLoc['name'].iat[0]
locationCode = searchLoc['code'].iat[0]


items_raw = fetch_all_records(
    "/inventory/items",
    records_key="items",
    query='effectiveLocationId=='+locationId
) 
print(f"{len(items_raw)} items found for with an effective location of: {locationName}")
items_df = pd.DataFrame(items_raw)

## 6. Inspect Your Data
Examine the first few rows of data to verify it's the location you'd like to use


In [ ]:
items_df.head()

## 7. Simplify the Data
A dataframe will include many columns that you won't want. You can create a new dataframe by extracting only the columns you want into a new set. 

In [ ]:
flattened_items = pd.DataFrame()
flattened_items['barcode'] = items_df['barcode']

# Pull the 'prefix' key out of each row's nested dictionary, or return None if it doesn't exist
flattened_items['prefix'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('prefix') if isinstance(x,dict) else None)
flattened_items['callNumber'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('callNumber') if isinstance(x,dict) else None)
flattened_items['suffix'] = items_df['effectiveCallNumberComponents'].apply(lambda x: x.get('suffix') if isinstance(x,dict) else None)
flattened_items['materialTypeId']  =  items_df['materialType'].apply(lambda x: x.get('name') if isinstance(x,dict) else None)
flattened_items['status']  =  items_df['status'].apply(lambda x: x.get('name') if isinstance(x,dict) else None)
flattened_items['title'] = items_df['title']

flattened_items.head()

## 8. Write to File

In [ ]:

today = date.today().strftime("%Y-%m-%d")
filename = f"{today}-{locationCode}-shelflist.csv"
flattened_items.to_csv(filename, index=False)
print(f"Data written to {filename}")